# Clustering based on clock genes (or other pre-determined gene list)

In [ ]:
dir_notebook = '../notebook'
name_dir = 'circa-SD'

df = pd.read_parquet(f'{dir_notebook}/csv/circa-SD/circa-SD_norm_combined.parquet')

In [ ]:
df_sub = df[df['cell_type_final'] == "Astro TE"]
df_sub.head()

In [ ]:
from module.misc import genes_list

clock_genes = genes_list("clock")

# df_sub = df_sub.filter(clock_genes, axis = 1)
# df_sub.index

In [ ]:
from sklearn.preprocessing import StandardScaler

pca = PCA(n_components=2)

x = df_sub.loc[:, clock_genes].values
x = StandardScaler().fit_transform(x)

principalComponents = pca.fit_transform(x)

finalDf = pd.DataFrame(data = principalComponents , columns = ['principal component 1', 'principal component 2'])

X_pca = principalComponents
kmeans = KMeans(n_clusters=2,
                random_state=0).fit(X_pca)

finalDf['cluster'] = kmeans.labels_
finalDf['cell_id'] = df_sub.index

In [ ]:
fig = plt.figure(figsize = (8,8))
ax = fig.add_subplot(1,1,1) 
ax.set_xlabel('Principal Component 1', fontsize = 15)
ax.set_ylabel('Principal Component 2', fontsize = 15)
ax.set_title('2 component PCA', fontsize = 20)

ax.scatter(finalDf['principal component 1'], finalDf['principal component 2'], c = finalDf['cluster'], s = 1)
ax.grid()

In [ ]:
df_sub['cluster'] = df_sub.index.map(dict(zip(finalDf['cell_id'],finalDf['cluster'])))
df_sub['x_centroid'] = df_sub.index.map(dict(zip(df['cell_id'],df['x_centroid'])))
df_sub['y_centroid'] = df_sub.index.map(dict(zip(df['cell_id'],df['y_centroid'])))
df_sub['sample'] = df_sub.index.map(dict(zip(df['cell_id'],df['sample'])))
df_sub['sample'].unique()

In [ ]:
plt.figure(figsize = (16,10))
plt.scatter(df_sub[df_sub['sample']=='circa4-IGM-ZT01']['x_centroid'], df_sub[df_sub['sample']=='circa4-IGM-ZT01']['y_centroid'],c=df_sub[df_sub['sample']=='circa4-IGM-ZT01']['cluster'], s = 1)
plt.colorbar()

In [ ]:
import umap
reducer = umap.UMAP()
embedding = reducer.fit_transform(x)
embedding.shape

In [ ]:
finalDf_umap = pd.DataFrame(data = embedding , columns = ['principal component 1', 'principal component 2'])

X_pca = embedding
kmeans = KMeans(#n_clusters=2,
                random_state=0).fit(X_pca)

finalDf_umap['cluster'] = kmeans.labels_
finalDf_umap['cell_id'] = df_sub.index

In [ ]:
fig = plt.figure(figsize = (8,8))
ax = fig.add_subplot(1,1,1) 
ax.set_xlabel('Principal Component 1', fontsize = 15)
ax.set_ylabel('Principal Component 2', fontsize = 15)
ax.set_title('UMAP', fontsize = 20)

ax.scatter(finalDf_umap['principal component 1'], finalDf_umap['principal component 2'], c = finalDf_umap['cluster'], s = 1)
ax.grid()

In [ ]:
df_sub['cluster_umap'] = df_sub.index.map(dict(zip(finalDf_umap['cell_id'],finalDf_umap['cluster'])))
df_sub['x_centroid'] = df_sub.index.map(dict(zip(df['cell_id'],df['x_centroid'])))
df_sub['y_centroid'] = df_sub.index.map(dict(zip(df['cell_id'],df['y_centroid'])))
df_sub['sample'] = df_sub.index.map(dict(zip(df['cell_id'],df['sample'])))
df_sub['sample'].unique()

In [ ]:
plt.figure(figsize = (16,10))
plt.scatter(df_sub[df_sub['sample']=='circa4-IGM-ZT01']['x_centroid'], df_sub[df_sub['sample']=='circa4-IGM-ZT01']['y_centroid'],c=df_sub[df_sub['sample']=='circa4-IGM-ZT01']['cluster_umap'], s = 3)
plt.colorbar()

### Filter from adata

In [ ]:
clock_genes

In [ ]:
adata_subcluster = adata[:, adata.var_names.isin(clock_genes)].copy()

In [ ]:
adata_subcluster.obs['sample'].unique()

In [ ]:
adata_subcluster = adata_subcluster[adata_subcluster.obs['cell_type_final']=="Oligodendrocyte"]
adata_subcluster = adata_subcluster[adata_subcluster.obs['sample']=='circa4-IGM-ZT01']
adata_subcluster.shape

In [ ]:
import scanpy as sc

sc.pp.pca(adata_subcluster)
sc.pp.neighbors(adata_subcluster)
sc.tl.umap(adata_subcluster)

In [ ]:
# extract pca coordinates
X_pca = adata_subcluster.obsm['X_pca'] 

### Kmeans clustering
### You can choose the number of clusters by uncommenting n_clusters option
# kmeans = KMeans(#n_clusters=4,
#                 random_state=0).fit(X_pca) 
# adata_subcluster.obs['kmeans'] = kmeans.labels_.astype(str)

sc.tl.leiden(adata_subcluster, resolution = 0.1)

In [ ]:
clustering_method = 'leiden'

In [ ]:
from matplotlib.pyplot import rc_context
with rc_context({"figure.figsize": (5, 5)}):
    sc.pl.umap(
        adata_subcluster,
        color=clustering_method,
        add_outline=False,
        legend_loc="on data",
        legend_fontsize=12,
        legend_fontoutline=2,
        frameon=False,
        palette="tab20",
    )
sc.pl.pca(adata_subcluster,
         color=clustering_method,
         palette="tab20",
         )

In [ ]:
### Number of cells per clusters
max_clust = len(adata_subcluster.obs[clustering_method].unique())
for i in range(0, max_clust):
    count = adata_subcluster.obs[clustering_method].value_counts().iloc[i]
    print(f"Cluster {i} : {count} cells")

# adata_subcluster.obs['leiden'].sample(10)

In [ ]:
from module.misc import sample_name_import

name_dir = "circa-SD"

samples, samples_ids = sample_name_import(name_dir)

print(len(samples))
print(samples)

In [ ]:
import seaborn as sns

### Generate a color palette for the clusters - to make color stay consistent across samples
adata_subcluster.obs[clustering_method] = adata_subcluster.obs[clustering_method].astype(int)

# Create a palette with a unique color for each cluster
num_clusters = len(adata_subcluster.obs[clustering_method].unique())
palette = sns.color_palette("tab20", n_colors=num_clusters)

# Map each 'leiden' value to a color
adata_subcluster.obs['kmeans_colors'] = adata_subcluster.obs[clustering_method].apply(lambda x: palette[x])

# Mapping of clusters
fig, axs = plt.subplots(4,3,figsize=(15, 20))
axs = axs.flatten()
clusters_plot = {
    # 0: 'orchid', 1: 'forestgreen', 2: 'black', 3:'red', 4:'cyan', 5:'blue', 6:'darkorange',7:'coral',
    # 8:'forestgreen', 9: 'coral',10:'red', 11:'cyan',
    6:'blue',0:'darkorange', 14:'black'
}

for idx, sample in enumerate(samples_ids):
    adata_sel = adata_subcluster[(adata_subcluster.obs['sample'] == sample)]
    for cluster_id in adata_sel.obs[clustering_method].unique():
        cluster_data = adata_sel.obs[adata_sel.obs[clustering_method] == cluster_id]
        colors = clusters_plot[cluster_id] if cluster_id in clusters_plot else "none"
        colors= cluster_data['kmeans_colors'].unique()[0]
        axs[idx].scatter(cluster_data['x_centroid'], cluster_data['y_centroid'], color=colors, s=0.1, label=cluster_id)
        axs[idx].set_title(f"Sample {sample}")

In [ ]:
### Correlation map of subclusters
cont_tab = pd.crosstab(adata_subcluster.obs[clustering_method], adata_subcluster.obs['circascore'], normalize="index")
cont_tab = cont_tab.loc[:, cont_tab.sum(axis=0) > 0.05]
plt.figure(figsize=(15, 15))
sns.heatmap(cont_tab.T, annot=True, cmap="YlGnBu", fmt=".1f", cbar=False) 

In [ ]:
df = pd.DataFrame(data=adata_subcluster.X.toarray(), index=adata_subcluster.obs_names, columns=adata_subcluster.var_names)

from module.xenium_preprocessing import add_annotations
df = add_annotations(adata_subcluster,df)
# name_dir = 'circa-SD'